In [0]:
%python
dailyPricingStreamingSourceFolderPath = "abfss://bronze@adldatalakehouseyubai.dfs.core.windows.net/daily-pricing-streaming-source-data"

In [0]:
USE CATALOG pricing_analytics;


In [0]:
%python
dailyPricingSourceStreamDF = spark.sql("""
SELECT
current_timestamp() as DATETIME_OF_PRICING,
try_cast(ROW_ID as BIGINT) as ROW_ID,
STATE_NAME,
MARKET_NAME,
PRODUCTGROUP_NAME,
PRODUCT_NAME,
VARIETY,
ORIGIN,
try_cast(ARRIVAL_IN_TONNES as DECIMAL(18,2)) 
    + extract(seconds from current_timestamp()) / 100 as ARRIVAL_IN_TONNES,
try_cast(MINIMUM_PRICE as DECIMAL(36,2)) 
    + extract(seconds from current_timestamp()) / 100 as MINIMUM_PRICE,
try_cast(MAXIMUM_PRICE as DECIMAL(36,2)) 
    + extract(seconds from current_timestamp()) / 100 as MAXIMUM_PRICE,
try_cast(MODAL_PRICE as DECIMAL(36,2)) 
    + extract(seconds from current_timestamp()) / 100 as MODAL_PRICE,
current_timestamp() as source_stream_load_datetime
FROM bronze.daily_pricing
WHERE DATE_OF_PRICING = '01/01/2023'
""")

In [0]:
%python

(dailyPricingSourceStreamDF
.write
.mode('append')
.json(dailyPricingStreamingSourceFolderPath)
)

In [0]:
%python
dbutils.fs.ls(dailyPricingStreamingSourceFolderPath)